# Buổi 4: Thu thập dữ liệu
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Đọc dữ liệu từ Excel nhiều sheet và JSON
- Truy vấn cơ sở dữ liệu SQL từ Python
- Gọi API web bằng thư viện requests
- Thu thập dữ liệu từ trang web bằng BeautifulSoup và pandas.read_html
- Gộp dữ liệu từ nhiều nguồn vào một DataFrame

**Dữ liệu dùng trong buổi:** `san_pham.xlsx`, `cua_hang.db`, `ty_gia_api.json`, `cua_hang_mau.html` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [15]:
%pip install requests beautifulsoup4 openpyxl lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import json, sqlite3
from io import StringIO
import pandas as pd
import requests
from bs4 import BeautifulSoup
DATA = "../data/"

> **Lưu ý pháp lý và đạo đức:** chỉ thu thập dữ liệu công khai, đọc điều khoản sử dụng và tệp `robots.txt` của website,
> không gửi yêu cầu dồn dập (nên `time.sleep()` giữa các lần), không thu thập dữ liệu cá nhân trái phép
> (Nghị định 13/2023/NĐ-CP về bảo vệ dữ liệu cá nhân).

## Phần A. Bài tập có hướng dẫn

### A1. Excel nhiều sheet
`sheet_name=None` đọc tất cả sheet thành một dict `{tên sheet: DataFrame}`.

In [17]:
sheets = pd.read_excel(DATA + "san_pham.xlsx", sheet_name=None)
print("Các sheet:", list(sheets))
sp = sheets["SanPham"]
ncc = sheets["NhaCungCap"]
sp.head()

Các sheet: ['SanPham', 'NhaCungCap']


,ma_sp,ten_sp,nhom_hang,gia_ban,ton_kho,ma_ncc
0,SP001,Sản phẩm 1,Sức khỏe & Làm đẹp,872000,27,NCC3
1,SP002,Sản phẩm 2,Sức khỏe & Làm đẹp,587000,231,NCC2
2,SP003,Sản phẩm 3,Điện tử,647000,125,NCC3
3,SP004,Sản phẩm 4,Gia dụng,306000,114,NCC2
4,SP005,Sản phẩm 5,Thể thao & Du lịch,408000,58,NCC2


In [18]:
# Ghép sản phẩm với nhà cung cấp theo khóa chung ma_ncc
sp_ncc = sp.merge(ncc, on="ma_ncc", how="left")
sp_ncc[["ma_sp", "ten_sp", "gia_ban", "ten_ncc", "thoi_gian_giao_ngay"]].head()

,ma_sp,ten_sp,gia_ban,ten_ncc,thoi_gian_giao_ngay
0,SP001,Sản phẩm 1,872000,Công ty Đại Việt,5
1,SP002,Sản phẩm 2,587000,Công ty An Khang,3
2,SP003,Sản phẩm 3,647000,Công ty Đại Việt,5
3,SP004,Sản phẩm 4,306000,Công ty An Khang,3
4,SP005,Sản phẩm 5,408000,Công ty An Khang,3


### A2. Truy vấn SQL từ Python
Doanh nghiệp thường lưu dữ liệu trong CSDL (MySQL, SQL Server, PostgreSQL...). Ở đây dùng SQLite (một tệp `.db`),
cách làm giống hệt: kết nối → viết câu SQL → `pd.read_sql()`.

In [19]:
con = sqlite3.connect(DATA + "cua_hang.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))
pd.read_sql("SELECT * FROM don_hang LIMIT 5", con)

         name
0  khach_hang
1    don_hang


,ma_don,ngay_dat,ma_kh,kenh,nhom_hang,san_pham,so_luong,don_gia,trang_thai
0,DH002819,2025-07-01,KH0448,App,Mỹ phẩm,Mỹ phẩm D,2,395000,Đã hủy
1,DH003115,2025-07-01,KH0498,App,Laptop,Laptop E,2,16609000,Hoàn thành
2,DH000716,2025-07-01,KH0110,App,Phụ kiện,Phụ kiện E,2,434000,Hoàn thành
3,DH001141,2025-07-01,KH0186,Website,Thời trang,Thời trang D,2,615000,Hoàn thành
4,DH000852,2025-07-01,KH0135,Website,Gia dụng,Gia dụng A,2,1343000,Hoàn thành


In [21]:
sql = """
SELECT k.vung, COUNT(*) AS so_don, SUM(d.so_luong * d.don_gia) / 1e6 AS doanh_thu_trieu
FROM don_hang d JOIN khach_hang k ON d.ma_kh = k.ma_kh
WHERE d.trang_thai = 'Hoàn thành'
GROUP BY k.vung
ORDER BY doanh_thu_trieu DESC
"""
pd.read_sql(sql, con).round(1)

,vung,so_don,doanh_thu_trieu
0,Miền Nam,662,3691.3
1,Miền Bắc,521,3040.1
2,Miền Trung,307,1939.2


In [22]:
# Truyền tham số an toàn bằng dấu ? (tránh lỗi SQL injection)
pd.read_sql("SELECT * FROM don_hang WHERE kenh = ? AND ngay_dat >= ?", con, params=("App", "2025-12-01")).head()

,ma_don,ngay_dat,ma_kh,kenh,nhom_hang,san_pham,so_luong,don_gia,trang_thai
0,DH002348,2025-12-01,KH0360,App,Điện thoại,Điện thoại A,2,8649000,Đã hủy
1,DH002376,2025-12-01,KH0362,App,Thời trang,Thời trang C,3,639000,Hoàn thành
2,DH002146,2025-12-02,KH0333,App,Mỹ phẩm,Mỹ phẩm D,1,556000,Hoàn thành
3,DH000119,2025-12-02,KH0020,App,Laptop,Laptop C,1,20053000,Đã hủy
4,DH001173,2025-12-02,KH0196,App,Mỹ phẩm,Mỹ phẩm B,2,461000,Hoàn thành


### A3. Gọi API
API trả dữ liệu dạng JSON. Quy trình: `requests.get(url, params=..., timeout=...)` → kiểm tra `status_code` → `.json()`.
Ô dưới thử gọi một API tỷ giá miễn phí; nếu không có mạng sẽ dùng tệp mẫu `ty_gia_api.json`.

In [23]:
try:
    r = requests.get("https://open.er-api.com/v6/latest/USD", timeout=5)
    r.raise_for_status()
    du_lieu = r.json()
    print("Nguồn: API trực tuyến –", du_lieu.get("time_last_update_utc"))
    print("1 USD =", du_lieu["rates"]["VND"], "VND")
except Exception as e:
    print("Không gọi được API (", type(e).__name__, ") → dùng dữ liệu mẫu")

Nguồn: API trực tuyến – Sat, 07 Nov 2026 00:02:31 +0000
1 USD = 25585.350173 VND


In [24]:
with open(DATA + "ty_gia_api.json", encoding="utf-8") as f:
    api = json.load(f)
print(api.keys())
print(api["rates"][0])

dict_keys(['source', 'base', 'rates'])
{'date': '2025-10-01', 'rates': {'USD': 25328.92, 'EUR': 27337.66, 'JPY': 169.08, 'CNY': 3515.09, 'KRW': 18.51, 'SGD': 19667.48}}


JSON lồng nhau → bảng phẳng bằng `pd.json_normalize`:

In [25]:
ty_gia = pd.json_normalize(api["rates"])
ty_gia.columns = [c.replace("rates.", "") for c in ty_gia.columns]
ty_gia["date"] = pd.to_datetime(ty_gia["date"])
ty_gia.head()

,date,USD,EUR,JPY,CNY,KRW,SGD
0,2025-10-01,25328.92,27337.66,169.08,3515.09,18.51,19667.48
1,2025-10-02,25490.56,27716.17,168.31,3534.55,18.40,19740.49
2,2025-10-03,25271.11,27465.45,167.91,3530.63,18.36,19604.51
3,2025-10-04,25419.11,27428.08,167.91,3518.37,18.43,19592.73
4,2025-10-05,25400.11,27458.10,167.81,3521.25,18.35,19465.16


### A4. Thu thập dữ liệu từ trang web (web scraping)
Bước 1: lấy mã HTML (trên web thật: `requests.get(url).text`; ở đây đọc tệp mẫu).
Bước 2: dùng BeautifulSoup tìm phần tử theo thẻ và class (xem bằng chuột phải → *Inspect* trên trình duyệt).

In [26]:
with open(DATA + "cua_hang_mau.html", encoding="utf-8") as f:
    html = f.read()
soup = BeautifulSoup(html, "html.parser")
print(soup.title.text)
the_dau = soup.select_one("div.product")
print(the_dau.prettify()[:400])

Cửa hàng Phụ kiện Mẫu
<div class="product" data-id="P001">
 <h3 class="name">
  Tai nghe Bluetooth
 </h3>
 <span class="price">
  2.935.000₫
 </span>
 <span class="old-price">
  4.372.000₫
 </span>
 <div class="rating">
  3.3
 </div>
 <div class="sold">
  Đã bán 2008
 </div>
 <div class="stock">
  Còn hàng
 </div>
</div>



In [27]:
def so(chuoi):
    """'1.250.000₫' -> 1250000"""
    return int("".join(ch for ch in chuoi if ch.isdigit()))

ds = []
for p in soup.select("div.product"):
    ds.append({
        "ma": p["data-id"],
        "ten": p.select_one(".name").text.strip(),
        "gia": so(p.select_one(".price").text),
        "gia_cu": so(p.select_one(".old-price").text),
        "danh_gia": float(p.select_one(".rating").text),
        "da_ban": so(p.select_one(".sold").text),
        "con_hang": p.select_one(".stock").text == "Còn hàng",
    })
san_pham_web = pd.DataFrame(ds)
san_pham_web["giam_%"] = ((1 - san_pham_web["gia"] / san_pham_web["gia_cu"]) * 100).round(1)
san_pham_web.head()

,ma,ten,gia,gia_cu,danh_gia,da_ban,con_hang,giam_%
0,P001,Tai nghe Bluetooth,2935000,4372000,3.3,2008,True,32.9
1,P002,Sạc dự phòng 10000mAh,3311000,3653000,3.5,3935,False,9.4
2,P003,Chuột không dây,2025000,2838000,3.3,3748,False,28.6
3,P004,Bàn phím cơ,2684000,3868000,4.0,4458,True,30.6
4,P005,Loa mini,1119000,1323000,4.5,849,True,15.4


Bảng HTML (`<table>`) đọc nhanh bằng `pd.read_html` – trả về list các DataFrame:

In [28]:
bang = pd.read_html(StringIO(html), attrs={"id": "shipping"})[0]
bang

,Khu vực,Phí (nghìn đồng),Thời gian (ngày)
0,Nội thành TP.HCM,15,1
1,Ngoại thành TP.HCM,25,2
2,Miền Nam,30,2
3,Miền Trung,35,3
4,Miền Bắc,40,4


### A5. Gộp nhiều nguồn và lưu kết quả
Quy đổi giá sản phẩm thu thập từ web sang USD theo tỷ giá trung bình tháng từ API, rồi lưu thành CSV.

In [29]:
usd_tb = ty_gia["USD"].mean()
san_pham_web["gia_usd"] = (san_pham_web["gia"] / usd_tb).round(2)
san_pham_web.to_csv("san_pham_thu_thap.csv", index=False, encoding="utf-8-sig")
print(f"Tỷ giá TB: {usd_tb:,.0f} VND/USD")
san_pham_web[["ten", "gia", "gia_usd"]].head()

Tỷ giá TB: 25,357 VND/USD


,ten,gia,gia_usd
0,Tai nghe Bluetooth,2935000,115.75
1,Sạc dự phòng 10000mAh,3311000,130.58
2,Chuột không dây,2025000,79.86
3,Bàn phím cơ,2684000,105.85
4,Loa mini,1119000,44.13


## Phần B. Bài tập tự làm

### Bài B1. Hàng sắp hết và nhà cung cấp

Từ `san_pham.xlsx`: liệt kê sản phẩm có tồn kho dưới 30, kèm tên nhà cung cấp, tỉnh thành và thời gian giao hàng.
Sắp xếp theo tồn kho tăng dần. Nhà cung cấp nào có nhiều sản phẩm sắp hết nhất?

In [30]:
# Lọc sản phẩm có tồn kho dưới 30
sap_het = sp_ncc[sp_ncc["ton_kho"] < 30]

# Sắp xếp tồn kho từ thấp đến cao
sap_het = sap_het.sort_values("ton_kho")

# Hiển thị thông tin
print(sap_het[[
    "ten_sp", "ton_kho", "ten_ncc",
    "tinh_thanh", "thoi_gian_giao_ngay"
]])

# Đếm số sản phẩm sắp hết theo nhà cung cấp
thong_ke = sap_het["ten_ncc"].value_counts()

print("\nSố sản phẩm sắp hết theo nhà cung cấp:")
print(thong_ke)

print("\nNhà cung cấp có nhiều sản phẩm sắp hết nhất:")
print(thong_ke[thong_ke == thong_ke.max()])


         ten_sp  ton_kho           ten_ncc tinh_thanh  thoi_gian_giao_ngay
16  Sản phẩm 17        3   Công ty Sao Mai    Đà Nẵng                    4
39  Sản phẩm 40       15  Công ty Đại Việt     Hà Nội                    5
9   Sản phẩm 10       24   Công ty Sao Mai    Đà Nẵng                    4
0    Sản phẩm 1       27  Công ty Đại Việt     Hà Nội                    5

Số sản phẩm sắp hết theo nhà cung cấp:
ten_ncc
Công ty Sao Mai     2
Công ty Đại Việt    2
Name: count, dtype: int64

Nhà cung cấp có nhiều sản phẩm sắp hết nhất:
ten_ncc
Công ty Sao Mai     2
Công ty Đại Việt    2
Name: count, dtype: int64


### Bài B2. Viết truy vấn SQL

Từ `cua_hang.db`, viết câu SQL (dùng `pd.read_sql`) để lấy: doanh thu (triệu đồng) và số đơn **hoàn thành** theo **kênh bán**,
sắp xếp giảm dần. Sau đó viết truy vấn thứ hai lấy 10 khách hàng có tổng chi tiêu cao nhất (mã KH, họ tên, vùng, tổng chi tiêu).

In [31]:
# Câu 1: Doanh thu và số đơn hoàn thành theo kênh bán
sql1 = """
SELECT kenh,
       COUNT(*) AS so_don,
       SUM(so_luong * don_gia) / 1000000.0 AS doanh_thu_trieu
FROM don_hang
WHERE trang_thai = 'Hoàn thành'
GROUP BY kenh
ORDER BY doanh_thu_trieu DESC
"""

print("Doanh thu theo kênh bán:")
print(pd.read_sql(sql1, con).round(2))

# Câu 2: Top 10 khách hàng chi tiêu nhiều nhất
sql2 = """
SELECT k.ma_kh, k.ho_ten, k.vung,
       SUM(d.so_luong * d.don_gia) AS tong_chi_tieu
FROM don_hang d
JOIN khach_hang k ON d.ma_kh = k.ma_kh
WHERE d.trang_thai = 'Hoàn thành'
GROUP BY k.ma_kh, k.ho_ten, k.vung
ORDER BY tong_chi_tieu DESC
LIMIT 10
"""

print("\nTop 10 khách hàng chi tiêu cao nhất:")
print(pd.read_sql(sql2, con))


Doanh thu theo kênh bán:
       kenh  so_don  doanh_thu_trieu
0       App     449          2502.70
1   Website     399          2412.05
2  Sàn TMĐT     418          2397.33
3  Cửa hàng     224          1358.58

Top 10 khách hàng chi tiêu cao nhất:
    ma_kh          ho_ten        vung  tong_chi_tieu
0  KH0475  Khách hàng 475    Miền Bắc      196091000
1  KH0455  Khách hàng 455    Miền Bắc      170637000
2  KH0424  Khách hàng 424  Miền Trung      151071000
3  KH0417  Khách hàng 417    Miền Bắc      143247000
4  KH0020   Khách hàng 20  Miền Trung      132441000
5  KH0264  Khách hàng 264    Miền Bắc      127181000
6  KH0331  Khách hàng 331  Miền Trung      125428000
7  KH0087   Khách hàng 87    Miền Bắc      121957000
8  KH0269  Khách hàng 269  Miền Trung      118968000
9  KH0224  Khách hàng 224    Miền Nam      117175000


### Bài B3. Phân tích tỷ giá

Từ DataFrame `ty_gia` (Phần A3): tìm ngày USD cao nhất và thấp nhất trong tháng, biên độ dao động (%),
và tính % thay đổi của mỗi đồng tiền từ đầu tháng đến cuối tháng.

In [32]:
# Sắp xếp theo ngày
ty_gia = ty_gia.sort_values("date")

# Tìm tỷ giá USD cao nhất và thấp nhất
cao_nhat = ty_gia.loc[ty_gia["USD"].idxmax()]
thap_nhat = ty_gia.loc[ty_gia["USD"].idxmin()]

print("USD cao nhất:", cao_nhat["USD"],
      "- Ngày:", cao_nhat["date"].date())

print("USD thấp nhất:", thap_nhat["USD"],
      "- Ngày:", thap_nhat["date"].date())

# Tính biên độ dao động
bien_do = (cao_nhat["USD"] - thap_nhat["USD"]) / thap_nhat["USD"] * 100

print("Biên độ dao động: %.2f%%" % bien_do)

# Tính tỷ lệ thay đổi đầu tháng và cuối tháng
print("\nTỷ lệ thay đổi của từng đồng tiền:")

for cot in ty_gia.columns:
    if cot != "date":
        dau_thang = ty_gia[cot].iloc[0]
        cuoi_thang = ty_gia[cot].iloc[-1]

        thay_doi = (cuoi_thang - dau_thang) / dau_thang * 100

        print(f"{cot}: {thay_doi:.2f}%")


USD cao nhất: 25497.31 - Ngày: 2025-10-24
USD thấp nhất: 25180.84 - Ngày: 2025-10-08
Biên độ dao động: 1.26%

Tỷ lệ thay đổi của từng đồng tiền:
USD: 0.39%
EUR: 0.28%
JPY: 0.35%
CNY: 0.09%
KRW: -0.97%
SGD: 0.07%


### Bài B4. Phân tích sản phẩm thu thập từ web

Từ `san_pham_web`: (a) 5 sản phẩm bán chạy nhất; (b) sản phẩm giảm giá sâu nhất; (c) điểm đánh giá trung bình của
sản phẩm còn hàng và hết hàng; (d) ước tính doanh thu = giá × số đã bán, sản phẩm nào cao nhất?

In [33]:
# a. Top 5 sản phẩm bán chạy nhất
print("5 sản phẩm bán chạy nhất:")
print(san_pham_web.nlargest(5, "da_ban")[
    ["ten", "gia", "da_ban"]
])

# b. Sản phẩm giảm giá sâu nhất
print("\nSản phẩm giảm giá sâu nhất:")
print(san_pham_web.nlargest(1, "giam_%")[
    ["ten", "gia", "gia_cu", "giam_%"]
])

# c. Điểm đánh giá trung bình theo tình trạng hàng
print("\nĐiểm đánh giá trung bình:")
print(san_pham_web.groupby("con_hang")["danh_gia"].mean().round(2))

# d. Ước tính doanh thu
san_pham_web["doanh_thu"] = (
    san_pham_web["gia"] * san_pham_web["da_ban"]
)

print("\nSản phẩm có doanh thu ước tính cao nhất:")
print(san_pham_web.nlargest(1, "doanh_thu")[
    ["ten", "gia", "da_ban", "doanh_thu"]
])


5 sản phẩm bán chạy nhất:
                      ten      gia  da_ban
9               Webcam HD   116000    4860
3             Bàn phím cơ  2684000    4458
5      Đồng hồ thông minh  3039000    4347
1   Sạc dự phòng 10000mAh  3311000    3935
15          Quạt mini USB  1885000    3872

Sản phẩm giảm giá sâu nhất:
                  ten      gia   gia_cu  giam_%
0  Tai nghe Bluetooth  2935000  4372000    32.9

Điểm đánh giá trung bình:
con_hang
False    4.10
True     4.08
Name: danh_gia, dtype: float64

Sản phẩm có doanh thu ước tính cao nhất:
                  ten      gia  da_ban    doanh_thu
5  Đồng hồ thông minh  3039000    4347  13210533000


### Bài B5. Tính phí giao hàng

Dùng bảng phí vận chuyển `bang` (Phần A4). Viết hàm `phi_ship(khu_vuc, gia_tri_don)` trả về phí (nghìn đồng):
miễn phí nếu đơn ≥ 500 nghìn đồng, ngược lại lấy theo bảng. Thử với ("Miền Bắc", 320) và ("Nội thành TP.HCM", 650).

In [34]:
def phi_ship(khu_vuc, gia_tri_don):
    if gia_tri_don >= 500:
        return 0

    dong = bang[bang["Khu vực"] == khu_vuc]

    if len(dong) == 0:
        print("Không tìm thấy khu vực giao hàng")
        return None

    return dong["Phí (nghìn đồng)"].iloc[0]

print("Miền Bắc, đơn 320 nghìn:", phi_ship("Miền Bắc", 320), "nghìn đồng")
print("Nội thành TP.HCM, đơn 650 nghìn:",
      phi_ship("Nội thành TP.HCM", 650), "nghìn đồng")


Miền Bắc, đơn 320 nghìn: 40 nghìn đồng
Nội thành TP.HCM, đơn 650 nghìn: 0 nghìn đồng


## Phần C. Bài tập làm thêm

### Bài C1. Hàm thu thập có xử lý lỗi

Viết hàm `lay_json(url, tep_du_phong)` thử gọi API (timeout 5 giây, tối đa 3 lần, nghỉ 1 giây giữa các lần);
nếu thất bại thì đọc tệp dự phòng. Hàm trả về dict và in ra nguồn dữ liệu đã dùng.

In [37]:
def lay_json(url, tep_du_phong):
    for lan in range(3):
        try:
            r = requests.get(url, timeout=5)
            r.raise_for_status()

            print("Lấy dữ liệu thành công từ API")
            return r.json()

        except Exception:
            print("Lần", lan + 1, "không thành công")

            if lan < 2:
                time.sleep(1)

    print("Dùng dữ liệu từ tệp dự phòng")

    with open(tep_du_phong, encoding="utf-8") as f:
        return json.load(f)

# Thử gọi API tỷ giá
url = "https://open.er-api.com/v6/latest/USD"

du_lieu = lay_json(url, DATA + "ty_gia_api.json")

print("Kiểu dữ liệu:", type(du_lieu))


Lấy dữ liệu thành công từ API
Kiểu dữ liệu: <class 'dict'>


### Bài C2. Thu thập từ một trang web thật

Chọn một trang có bảng dữ liệu công khai (ví dụ trang Wikipedia "Danh sách đơn vị hành chính cấp tỉnh của Việt Nam").
Dùng `requests` + `pd.read_html` lấy bảng, làm sạch tên cột, và cho biết 5 tỉnh có dân số lớn nhất.
Ghi rõ URL và ngày thu thập trong notebook. (Cần kết nối Internet; nếu trang thay đổi cấu trúc, hãy tự điều chỉnh.)

In [39]:
from datetime import date

url = "https://en.wikipedia.org/wiki/Provinces_of_Vietnam"

print("Nguồn:", url)
print("Ngày thu thập:", date.today())

r = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0 (Educational project)"},
    timeout=10
)
r.raise_for_status()

# Đọc các bảng HTML
cac_bang = pd.read_html(StringIO(r.text))

# Tìm bảng có cột dân số
for b in cac_bang:
    b.columns = [
        " ".join(map(str, c)) if isinstance(c, tuple) else str(c)
        for c in b.columns
    ]

    cot_dan_so = [c for c in b.columns if "population" in c.lower()]

    if cot_dan_so:
        bang_tinh = b.copy()
        break
else:
    raise ValueError("Không tìm thấy bảng dân số")

# Chuẩn hóa tên cột
bang_tinh.columns = (
    bang_tinh.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

cot = cot_dan_so[0].strip().lower().replace(" ", "_")

# Chuyển dân số sang dạng số
bang_tinh[cot] = pd.to_numeric(
    bang_tinh[cot].astype(str).str.replace(",", ""),
    errors="coerce"
)

# Hiển thị 5 tỉnh/thành có dân số lớn nhất
print("\n5 tỉnh/thành có dân số lớn nhất:")
print(bang_tinh.nlargest(5, cot))


Nguồn: https://en.wikipedia.org/wiki/Provinces_of_Vietnam
Ngày thu thập: 2026-11-07

5 tỉnh/thành có dân số lớn nhất:
    seal_seal city/province_city/province type_type  map_number_map_number  \
28        NaN            Ho Chi Minh City      City                      2   
12        NaN                       Hanoi      City                      1   
29        NaN                    An Giang  Province                     32   
13        NaN                    Haiphong      City                      3   
26        NaN                    Đồng Nai      City                      7   

   administrative_center_administrative_center  \
28                                      Saigon   
12                                   Hoàn Kiếm   
29                                    Rạch Giá   
13                                 Thủy Nguyên   
26                                   Trấn Biên   

    land_area_(km²)[3]_land_area_(km²)[3]  population[4]_population[4]  \
28                                   6

### Bài C3. Lưu dữ liệu thu thập vào CSDL

Tạo CSDL mới `du_lieu_thu_thap.db`, ghi `san_pham_web` thành bảng `san_pham` và `ty_gia` thành bảng `ty_gia`
(`DataFrame.to_sql`). Sau đó viết truy vấn SQL lấy các sản phẩm còn hàng có giá dưới 500.000 đ, kèm giá USD
tính theo tỷ giá ngày cuối cùng trong bảng `ty_gia`.

In [40]:
# Tạo cơ sở dữ liệu mới
con_moi = sqlite3.connect("du_lieu_thu_thap.db")

# Lưu dữ liệu sản phẩm
san_pham_web.to_sql(
    "san_pham",
    con_moi,
    if_exists="replace",
    index=False
)

# Lưu dữ liệu tỷ giá
ty_gia.to_sql(
    "ty_gia",
    con_moi,
    if_exists="replace",
    index=False
)

print("Đã lưu dữ liệu vào database")

# Truy vấn sản phẩm còn hàng, giá dưới 500.000 đồng
sql = """
SELECT s.ten, s.gia,
       ROUND(s.gia / t.USD, 2) AS gia_usd
FROM san_pham s
CROSS JOIN (
    SELECT USD
    FROM ty_gia
    ORDER BY date DESC
    LIMIT 1
) t
WHERE s.con_hang = 1
AND s.gia < 500000
ORDER BY s.gia ASC
"""

ket_qua = pd.read_sql(sql, con_moi)

print("\nSản phẩm còn hàng có giá dưới 500.000 đồng:")
print(ket_qua)

# Đóng kết nối
con_moi.close()


Đã lưu dữ liệu vào database

Sản phẩm còn hàng có giá dưới 500.000 đồng:
         ten     gia  gia_usd
0  Webcam HD  116000     4.56
